In [3]:
# import kagglehub

# # Download latest version
# path = kagglehub.dataset_download("sora1874/splits-updated-223")

# print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/splits-updated-223


In [5]:
!rm -rf '/kaggle/working/bert'
!rm -rf '/kaggle/working/output'
!rm -rf '/kaggle/working/data'
!rm -rf '/kaglle/working/logs'

In [6]:
shutil.copytree('/kaggle/input/splits-updated-223/splits/MELD', '/kaggle/working/data/MELD')

'/kaggle/working/data/MELD'

In [7]:
os.makedirs(os.path.dirname('/kaggle/working/bert/bert_mlm_psych_model.pth'), exist_ok=True)
shutil.copy('/kaggle/input/bert-pretrained/bert/bert_mlm_psych_model.pth', '/kaggle/working/bert/bert_mlm_psych_model.pth')

'/kaggle/working/bert/bert_mlm_psych_model.pth'

In [8]:
import os
from datasets import Dataset, DatasetDict
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
from transformers import TrainingArguments, Trainer
import shutil
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.optim import AdamW
import numpy as np 

In [2]:
data_dir = '/kaggle/working/data/MELD'
output_dir = '/kaggle/working/output'
train_path = os.path.join(data_dir, "train/text")
val_path = os.path.join(data_dir, "val/text")
test_path = os.path.join(data_dir, "test/text")

In [3]:
def load_data_from_files(folder_path):
    data = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".txt"):
            label_part = filename.split("_")[-1].split(".")[0]
            file_path = os.path.join(folder_path, filename)
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read().strip()
                data.append({"text": text, "label": label_part})
    return data

train_data = load_data_from_files(train_path)
val_data = load_data_from_files(val_path)
test_data = load_data_from_files(test_path)

In [4]:
train_ds = Dataset.from_pandas(pd.DataFrame(train_data))
val_ds = Dataset.from_pandas(pd.DataFrame(val_data))
test_ds = Dataset.from_pandas(pd.DataFrame(test_data))

dataset = DatasetDict({
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
})

In [5]:
label_to_id = {
    "neutral": 0,
    "joy": 1,
    "sadness": 2,
    "anger": 3,
    "surprise": 4,
    "fear": 5,
    "disgust": 6,
}

def label_str_to_int(examples):
    return {"label": label_to_id[examples["label"]]}

dataset = dataset.map(label_str_to_int)

Map:   0%|          | 0/3856 [00:00<?, ? examples/s]

Map:   0%|          | 0/435 [00:00<?, ? examples/s]

Map:   0%|          | 0/1016 [00:00<?, ? examples/s]

In [6]:
from sklearn.metrics import accuracy_score
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return {"accuracy": accuracy_score(labels, predictions)}

In [20]:
from transformers import get_linear_schedule_with_warmup
os.environ["WANDB_DISABLED"] = "true"  
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=7)
model.classifier = torch.nn.Sequential(
    torch.nn.Dropout(0.5),
    torch.nn.Linear(model.config.hidden_size, model.config.num_labels)
).to(device)
model_dir = '/kaggle/working/bert/bert_mlm_psych_model.pth'
model.load_state_dict(torch.load(model_dir, map_location=device), strict=False)
model.to(device)

# for layer in model.bert.encoder.layer[:-2]:  
#     for param in layer.parameters():
#         param.requires_grad = False


tokenized_dataset = dataset.map(
    lambda x: tokenizer(x["text"], truncation=True, padding="max_length", max_length=128),
    batched=True
).shuffle(seed=42)
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")


training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    learning_rate=1e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    gradient_accumulation_steps=2,
    max_grad_norm=0.5,
    logging_steps=50,
    logging_strategy="epoch",
    eval_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    report_to="none",
)

optimizer = torch.optim.AdamW([
    {"params": model.bert.encoder.layer[:-4].parameters(), "lr": 1e-6},
    {"params": model.bert.encoder.layer[-4:-2].parameters(), "lr": 5e-6},
    {"params": model.bert.encoder.layer[-2:].parameters(), "lr": 3e-5},
    {"params": model.classifier.parameters(), "lr": 1e-4}
])

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None),
)
from transformers import EarlyStoppingCallback
trainer.add_callback(EarlyStoppingCallback(
    early_stopping_patience=3,  
    early_stopping_threshold=0.01
))
trainer.train()

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
<ipython-input-20-9270c0df4cfb>:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We rec

Map:   0%|          | 0/3856 [00:00<?, ? examples/s]

Map:   0%|          | 0/435 [00:00<?, ? examples/s]

Map:   0%|          | 0/1016 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Accuracy
1,1.783100,1.682009,0.393103
2,1.569000,1.560033,0.420690
3,1.462300,1.539224,0.445977
4,1.396100,1.468445,0.480460
5,1.342900,1.446360,0.466667
6,1.303000,1.406489,0.473563
7,1.262100,1.398154,0.485057
8,1.242000,1.403285,0.482759


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked t

TrainOutput(global_step=488, training_loss=1.420063925571129, metrics={'train_runtime': 462.4707, 'train_samples_per_second': 83.378, 'train_steps_per_second': 1.297, 'total_flos': 2029203551846400.0, 'train_loss': 1.420063925571129, 'epoch': 8.0})

In [21]:
test_results = trainer.evaluate(tokenized_dataset["test"])
print(f"Test set evaluation results: {test_results}")

/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Test set evaluation results: {'eval_loss': 1.4078285694122314, 'eval_accuracy': 0.5098425196850394, 'eval_runtime': 5.7518, 'eval_samples_per_second': 176.64, 'eval_steps_per_second': 11.127, 'epoch': 8.0}


In [34]:
os.makedirs("/kaggle/working/output/final", exist_ok=True)

In [35]:
torch.save(model.state_dict(), "/kaggle/working/output/final/bert_meld_finetune_model.pth")

In [27]:
# 保存模型
model.save_pretrained(os.path.join(output_dir,"meld_finetuned_model"))
tokenizer.save_pretrained(os.path.join(output_dir,"meld_finetuned_model"))

('/kaggle/working/output/meld_finetuned_model/tokenizer_config.json',
 '/kaggle/working/output/meld_finetuned_model/special_tokens_map.json',
 '/kaggle/working/output/meld_finetuned_model/vocab.txt',
 '/kaggle/working/output/meld_finetuned_model/added_tokens.json',
 '/kaggle/working/output/meld_finetuned_model/tokenizer.json')